<a href="https://colab.research.google.com/github/BaseballCV/BaseballCV/blob/main/notebooks/base_detector_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base Detector Retraining — dataset v2 (#183)

Retrains the base detector on **dataset v2**: the same 660 broadcast frames as v1, but with 80 recovered base annotations (machine-mined, then visually verified — see `datasets/yolo/base_detect_dataset_v2/README.md`) and a proper 90/10 train/val split, which v1 never had.

**What this notebook does** (Runtime > Run all; GPU runtime recommended):

1. Downloads the v1 images via `LoadTools` and the curated v2 labels from this repo
2. Assembles the YOLO-format dataset (classes: `base`, `distractor`)
3. Fine-tunes YOLOv11s from COCO-pretrained weights
4. Reports validation mAP per class and compares the old vs new model on the same split
5. Exports the trained weights for review

This addresses the retraining half of [#183](https://github.com/BaseballCV/BaseballCV/issues/183). The distractor class is intentionally tiny (3 verified boxes: shoe, leg, pitching rubber) — it is a seed for future rubber/line annotation, not a complete fix; the interim `BaseDetector` inference mitigations remain in place.

In [ ]:
!pip install -q "baseballcv>=0.1.31"

import shutil
from pathlib import Path

import matplotlib.pyplot as plt
from ultralytics import YOLO

from baseballcv.functions import LoadTools

WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
load_tools = LoadTools()
print("Setup OK")

## 1. Data: v1 images + curated v2 labels

In [ ]:
# v1 images (vault-hosted) and the current production weights, for comparison later
images_dir = Path(str(load_tools.load_dataset('base_detect_dataset'))) / "images" / "train"
old_weights = load_tools.load_model('base_detector')
print(f"{len(list(images_dir.glob('*')))} images | old weights at {old_weights}")

# curated v2 labels from this repo (sparse clone — labels only, fast)
from IPython import get_ipython
REPO = Path("/content/repo")
if not REPO.exists():
    get_ipython().system('git clone -q --depth 1 --filter=blob:none --sparse https://github.com/BaseballCV/BaseballCV.git "{REPO}"')
get_ipython().system('git -C "{REPO}" sparse-checkout set datasets/yolo/base_detect_dataset_v2')
V2 = REPO / "datasets/yolo/base_detect_dataset_v2"
print("v2 labels:", len(list((V2 / 'labels/train').glob('*.txt'))), "train +",
      len(list((V2 / 'labels/val').glob('*.txt'))), "val")

In [ ]:
# Assemble the YOLO dataset: images from the vault, labels from the curated set
DS = WORK / "base_detect_v2"
for split in ("train", "val"):
    (DS / "images" / split).mkdir(parents=True, exist_ok=True)
    (DS / "labels" / split).mkdir(parents=True, exist_ok=True)
    for lf in (V2 / "labels" / split).glob("*.txt"):
        stem = lf.stem
        src = next((p for p in images_dir.glob(stem + ".*")), None)
        assert src is not None, f"image not found for label {stem}"
        shutil.copy(src, DS / "images" / split / src.name)
        shutil.copy(lf, DS / "labels" / split / lf.name)
(DS / "data.yaml").write_text(
    f"path: {DS.resolve().as_posix()}\ntrain: images/train\nval: images/val\n"
    "names:\n  0: base\n  1: distractor\n")
print("dataset assembled at", DS)

## 2. Sanity-check a few labels

In [ ]:
import cv2
import numpy as np

val_imgs = sorted((DS / "images" / "val").glob("*"))
tiles = []
for p in val_imgs[:8]:
    img = cv2.imread(str(p)); ih, iw = img.shape[:2]
    lf = DS / "labels" / "val" / (p.stem + ".txt")
    for line in lf.read_text().splitlines():
        c, xc, yc, w, h = line.split()[:5]
        xc, yc, w, h = float(xc)*iw, float(yc)*ih, float(w)*iw, float(h)*ih
        color = (0, 255, 0) if int(c) == 0 else (0, 165, 255)
        cv2.rectangle(img, (int(xc-w/2), int(yc-h/2)), (int(xc+w/2), int(yc+h/2)), color, 2)
    tiles.append(cv2.resize(img, (426, 240)))
rows = [np.hstack(tiles[i:i+4]) for i in range(0, len(tiles), 4)]
plt.figure(figsize=(17, 5*len(rows)))
plt.imshow(np.vstack(rows)[:, :, ::-1]); plt.axis("off"); plt.show()

## 3. Train

Fine-tunes `yolo11s` (same architecture as the production `base_detector`) from COCO-pretrained weights so the comparison against the current model is apples-to-apples. Roughly 20 minutes on a T4 GPU for 100 epochs at this dataset size.

In [ ]:
model = YOLO("yolo11s.pt")  # auto-downloads COCO-pretrained weights
model.train(
    data=str(DS / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    name="base_detector_v2",
)
BEST = Path(model.trainer.best).resolve()
print("best weights:", BEST)

## 4. Evaluate: new model vs production model on the same val split

The production model has no `distractor` class and its own training data overlaps this split, so treat these numbers as a same-split sanity check rather than a held-out benchmark. The honest question is: does the new model detect bases at least as well while producing fewer false base detections?

In [ ]:
new_model = YOLO(str(BEST))
metrics = new_model.val(data=str(DS / "data.yaml"), verbose=False)
print("NEW  mAP50 (all):", round(float(metrics.box.map50), 4))
print("NEW  mAP50 base (class 0):", round(float(metrics.box.maps[0]), 4))

old_model = YOLO(old_weights)
old_metrics = old_model.val(data=str(DS / "data.yaml"), classes=[0], verbose=False)
print("OLD  mAP50 base (class 0):", round(float(old_metrics.box.map50), 4))

In [ ]:
# Side-by-side predictions on a few val images
fig, axes = plt.subplots(3, 2, figsize=(16, 13))
for ax_pair, p in zip(axes, [str(p) for p in val_imgs[:3]]):
    for ax, (label, m) in zip(ax_pair, [("OLD (production)", old_model), ("NEW (v2)", new_model)]):
        r = m.predict(p, conf=0.5, verbose=False)[0]
        ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(label); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Export weights

In [ ]:
shutil.copy(str(BEST), str(WORK / "base_detector_v2.pt"))
print("artifact:", WORK / "base_detector_v2.pt")
from google.colab import files
files.download(str(WORK / "base_detector_v2.pt"))

## Next steps after training

1. Download `base_detector_v2.pt` and get it to the maintainers (PR referencing #183, or the contacts in the README).
2. Maintainer review flow: check val mAP, run the `BaseDetector` false-positive regression (mound shots, slide footage), then publish to the vault as `base_detector_v2` — or replace `base_detector` if it strictly dominates.
3. Grow the distractor class: mine mound-visible frames from broadcast clips, auto-label rubber candidates with the retrained model, verify visually (same workflow used for the v2 labels — about 5 minutes per batch).